# AfriQBench Notebook 04 — Cross-Backend Comparison\n\nThis notebook runs the same four-qubit TFIM benchmark through three local execution classes: **ideal Qiskit Aer**, **controlled synthetic noise**, and **device-derived Aer noise from a cached IBM fake backend**.\n\nThe objective is to hold the scientific workload fixed while changing the execution environment.

## 1. Installation\n\n```bash\npython -m pip install -e ".[quantum,notebook]"\n```\n\nThe quantum extra includes `qiskit-ibm-runtime` because the device-derived path uses its cached fake-provider snapshots.

In [ ]:
from pathlib import Path\nimport sys\n\nrepo_root = Path.cwd()\nif not (repo_root / 'src').exists():\n    repo_root = repo_root.parent\nsys.path.insert(0, str(repo_root / 'src'))\n\nimport json\nimport platform\nimport importlib.metadata as metadata\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nfrom afriqbench.metrics import absolute_error, relative_error\nfrom afriqbench.models.tfim import tfim_hamiltonian\nfrom afriqbench.quantum.cross_backend import (\n    available_fake_backend_names,\n    run_cross_backend_comparison,\n)\nfrom afriqbench.quantum.qiskit_tfim import bind_ansatz, circuit_resource_metrics\nfrom afriqbench.reference.exact import ground_state\n

## 2. Record the software environment

In [ ]:
packages = ['afriqbench', 'qiskit', 'qiskit-aer', 'qiskit-ibm-runtime', 'numpy']\nversions = {'python': platform.python_version()}\nfor package in packages:\n    try:\n        versions[package] = metadata.version(package)\n    except metadata.PackageNotFoundError:\n        versions[package] = 'not installed as package'\nversions\n

## 3. Prepare the fixed TFIM workload\n\nAll targets use the same Hamiltonian, variational parameters, shot budget, and seed.

In [ ]:
n_qubits = 4\nJ = 1.0\nh = 1.0\nshots = 20_000\nseed = 12345\ndevice_id = 'fake_manila'\n\nbaseline_path = repo_root / 'results' / 'ideal_quantum_n4_h1_baseline.json'\nbaseline = json.loads(baseline_path.read_text(encoding='utf-8'))\nparameters = np.asarray(baseline['ansatz']['parameters'], dtype=float)\ncircuit = bind_ansatz(n_qubits, parameters, reps=2)\n\nH = tfim_hamiltonian(n_qubits, J=J, h=h, periodic=False)\nexact_energy, _ = ground_state(H)\nstatevector_energy = baseline['ideal_statevector']['variational_energy']\n\nprint('Exact energy:', exact_energy)\nprint('Variational statevector energy:', statevector_energy)\nprint('Raw circuit resources:', circuit_resource_metrics(circuit))\n

## 4. Verify the cached device target\n\nThe default MVP uses `fake_manila`, a five-qubit cached backend. The loader also accepts aliases such as `manila` and `ibm_manila`.

In [ ]:
available = available_fake_backend_names()\nprint('fake_manila available:', 'fake_manila' in available)\nprint('Number of cached fake backends:', len(available))\nprint('Example names:', available[:12])\n

## 5. Run the canonical three-way comparison

In [ ]:
comparison = run_cross_backend_comparison(\n    circuit,\n    n_qubits=n_qubits,\n    J=J,\n    h=h,\n    periodic=False,\n    shots=shots,\n    seed=seed,\n    device_id=device_id,\n    controlled_single_qubit_error=0.001,\n    controlled_two_qubit_error=0.01,\n    controlled_readout_error=0.01,\n)\n

## 6. Build a common comparison table

In [ ]:
labels = {\n    'ideal_aer': 'Ideal Aer',\n    'controlled_noise_aer': 'Controlled noise',\n    'device_derived_aer': f'Device-derived ({device_id})',\n}\n\nrows = []\nfor key, result in comparison.items():\n    rows.append({\n        'target': labels[key],\n        'energy': result['energy'],\n        'shot_uncertainty': result['uncertainty'],\n        'absolute_error_vs_exact': absolute_error(result['energy'], exact_energy),\n        'relative_error_vs_exact': relative_error(result['energy'], exact_energy),\n    })\n\ndf = pd.DataFrame(rows)\ndf\n

## 7. Compare energies\n\nError bars represent finite-shot standard error. They do not capture the systematic shift introduced by a noise model.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))\nax.errorbar(\n    df['target'],\n    df['energy'],\n    yerr=df['shot_uncertainty'],\n    marker='o',\n    linestyle='none',\n    capsize=4,\n)\nax.axhline(exact_energy, linestyle='--', linewidth=1, label='Exact reference')\nax.axhline(statevector_energy, linestyle=':', linewidth=1, label='Variational statevector')\nax.set_ylabel('Estimated TFIM energy')\nax.set_title('AfriQBench cross-backend TFIM comparison')\nax.tick_params(axis='x', rotation=15)\nax.legend()\nfig.tight_layout()\nplt.show()\n

## 8. Compare application-level error

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))\nax.bar(df['target'], df['absolute_error_vs_exact'])\nax.set_ylabel(r'$|E_{\\mathrm{estimate}}-E_{\\mathrm{exact}}|$')\nax.set_title('Application-level error across execution targets')\nax.tick_params(axis='x', rotation=15)\nfig.tight_layout()\nplt.show()\n

## 9. Inspect device-derived compilation cost\n\nThe cached backend brings a device topology and native operation set. AfriQBench records post-transpilation resources because routing can change depth and two-qubit gate counts.

In [ ]:
device_result = comparison['device_derived_aer']\nprint('Device metadata:')\nprint(json.dumps(device_result['device_metadata'], indent=2))\nprint('\\nPost-transpilation resources:')\nprint(json.dumps(device_result['transpiled_resources'], indent=2))\n

## 10. Term-level comparison\n\nA single energy score can conceal where degradation occurs, so the benchmark preserves every Hamiltonian-term estimate.

In [ ]:
term_frames = []\nfor key, result in comparison.items():\n    frame = pd.DataFrame(result['terms'])\n    frame['target'] = labels[key]\n    term_frames.append(frame)\nterm_df = pd.concat(term_frames, ignore_index=True)\nterm_df[['target', 'term', 'expectation', 'uncertainty']]\n

## 11. Save the cross-backend record

In [ ]:
results_dir = repo_root / 'results'\nresults_dir.mkdir(exist_ok=True)\n\ncsv_path = results_dir / 'cross_backend_n4_h1_run.csv'\njson_path = results_dir / 'cross_backend_n4_h1_run.json'\n\ndf.to_csv(csv_path, index=False)\njson_path.write_text(\n    json.dumps({\n        'benchmark_id': 'tfim-cross-backend-n4-h1-v0',\n        'software_versions': versions,\n        'exact_energy': exact_energy,\n        'variational_statevector_energy': statevector_energy,\n        'shots_per_term': shots,\n        'seed': seed,\n        'results': comparison,\n    }, indent=2),\n    encoding='utf-8',\n)\n\nprint(csv_path)\nprint(json_path)\n

## What this establishes\n\nAfriQBench now has a reproducible chain from exact classical reference to ideal finite-shot quantum execution, controlled noise, and device-derived local noise. The device-derived path uses the same public Qiskit mechanism that current Metriq-Gym uses for cached IBM fake backends: `FakeProviderForBackendV2` plus `AerSimulator.from_backend(...)`.\n\nThe next layer can focus on **error mitigation with Mitiq** and, after that, packaging the workload into the schema/benchmark conventions required for an upstream Metriq contribution.